# MobileNetV3Large

This notebook :-
- apply augmentation only to minority classes `DME` and `DRUSEN` (class indices `1` and `2`),
- **save augmented images directly into the train folder** (run once before training),
- keep MobileNetV3Large end-to-end trainable,
- avoid redundant normalization because `include_preprocessing=True` already handles MobileNetV3 input scaling.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.metrics import Precision, Recall, F1Score
from tensorflow.keras.utils import image_dataset_from_directory


BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)
NUM_CLASSES = 4
EPOCHS = 15
AUTOTUNE = tf.data.AUTOTUNE

TRAIN_DIR = "./train"
VAL_DIR = "./val"

# Directory order is expected to be:
# 0 -> CNV, 1 -> DME, 2 -> DRUSEN, 3 -> NORMAL
MINORITY_CLASS_IDS = tf.constant([1, 2], dtype=tf.int32)


## Save Augmented Images to Disk (run once)

This cell reads every original image from `./train/DME` and `./train/DRUSEN`,
generates `AUGMENT_COPIES` augmented variants per image, and writes them back
into the **same class folder** with an `aug_<n>_` prefix.

Re-running the cell is safe: files that already start with `aug_` are skipped,
so originals are never augmented a second time.

In [ ]:
import os
import numpy as np
from pathlib import Path
import tensorflow as tf
from tensorflow.keras import layers

# ── configuration ────────────────────────────────────────────────────────────
MINORITY_CLASS_DIRS = ["DME", "DRUSEN"]  
AUGMENT_COPIES      = 3                  # how many augmented images per original
SUPPORTED_EXTS      = {".jpg", ".jpeg", ".png", ".bmp", ".tiff"}
SAVE_FORMAT         = "jpeg"              
IMAGE_SIZE          = (224, 224)
TRAIN_DIR           = "./train"


augment = tf.keras.Sequential(
    [
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.10),
        layers.RandomZoom(height_factor=0.10, width_factor=0.10),
        layers.RandomBrightness(factor=0.20),
        layers.RandomContrast(factor=0.20),
    ],
    name="disk_augmentation",
)


def load_image(path: Path) -> tf.Tensor:
    """Load an image from disk and resize to IMAGE_SIZE, returning uint8."""
    raw   = tf.io.read_file(str(path))
    image = tf.image.decode_image(raw, channels=3, expand_animations=False)
    image = tf.image.resize(image, IMAGE_SIZE, method="bilinear")
    image = tf.cast(image, tf.uint8)
    return image


def save_image(image: tf.Tensor, save_path: Path) -> None:
    """Encode a uint8 tensor and write it to save_path."""
    if SAVE_FORMAT == "jpeg":
        encoded = tf.image.encode_jpeg(image, quality=95)
        ext = ".jpg"
    else:
        encoded = tf.image.encode_png(image)
        ext = ".png"
    tf.io.write_file(str(save_path.with_suffix(ext)), encoded)


total_saved = 0

for class_name in MINORITY_CLASS_DIRS:
    class_dir = Path(TRAIN_DIR) / class_name
    if not class_dir.is_dir():
        print(f"[WARN] Directory not found, skipping: {class_dir}")
        continue

    originals = [
        p for p in sorted(class_dir.iterdir())
        if p.suffix.lower() in SUPPORTED_EXTS
        and not p.stem.startswith("aug_")
    ]

    print(f"[{class_name}] Found {len(originals)} original images — "
          f"generating {AUGMENT_COPIES} augmented copy/copies each ...")

    for img_path in originals:
        image = load_image(img_path)                        
        image_f = tf.cast(image, tf.float32)                

        for n in range(AUGMENT_COPIES):
            aug_image_f = augment(tf.expand_dims(image_f, 0), training=True)
            aug_image_f = tf.squeeze(aug_image_f, 0)
            aug_image_f = tf.clip_by_value(aug_image_f, 0.0, 255.0)
            aug_image   = tf.cast(aug_image_f, tf.uint8)

            out_path = class_dir / f"aug_{n}_{img_path.stem}"
            save_image(aug_image, out_path)
            total_saved += 1

    all_images = [p for p in class_dir.iterdir() if p.suffix.lower() in SUPPORTED_EXTS]
    print(f"[{class_name}] Total images in folder after augmentation: {len(all_images)}")

print(f"\nDone. {total_saved} augmented images saved to disk.")


## data Pipelines

Because the augmented images are already on disk, the pipeline is now a
straightforward

In [ ]:
raw_training_set = image_dataset_from_directory(
    TRAIN_DIR,
    labels="inferred",
    label_mode="categorical",
    color_mode="rgb",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    shuffle=True,
    interpolation="bilinear",
    follow_links=False,
    crop_to_aspect_ratio=False,
)

raw_validation_set = image_dataset_from_directory(
    VAL_DIR,
    labels="inferred",
    label_mode="categorical",
    color_mode="rgb",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    shuffle=False,
    interpolation="bilinear",
    follow_links=False,
    crop_to_aspect_ratio=False,
)

class_names = raw_training_set.class_names
expected_class_names = ["CNV", "DME", "DRUSEN", "NORMAL"]
if class_names != expected_class_names:
    raise ValueError(
        f"Unexpected class order: {class_names}. Expected {expected_class_names} "
        "so that minority indices 1 and 2 map to DME and DRUSEN."
    )

print("Class names:", class_names)


def cast_images(images, labels):
    images = tf.cast(images, tf.float32)
    return images, labels


# MobileNetV3Large with include_preprocessing=True already applies the
# correct input preprocessing internally. Do not add Rescaling(1./255)
# or preprocess_input on top of it, because that would double-normalize.
training_set = (
    raw_training_set
    .map(cast_images, num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE)
)

validation_set = (
    raw_validation_set
    .map(cast_images, num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE)
)


In [ ]:
mobnet = MobileNetV3Large(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet",
    pooling="avg",
    include_preprocessing=True,
)
mobnet.trainable = True

model = models.Sequential(
    [
        mobnet,
        layers.Dense(NUM_CLASSES, activation="softmax"),
    ]
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss="categorical_crossentropy",
    metrics=[
        "accuracy",
        F1Score(average="macro", name="f1_score"),
        Precision(name="precision"),
        Recall(name="recall"),
    ],
)

model.summary()


In [ ]:
history = model.fit(
    training_set,
    validation_data=validation_set,
    epochs=15,
)
